<a href="https://colab.research.google.com/github/gabrielfauro-bee/Atividades_FMC2026/blob/main/Atividade_Extracao_PDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Etapa 1 — Instalar e importar as bibliotecas

Nesta etapa são carregadas as ferramentas necessárias:

- **pdfplumber**: extrai o texto do arquivo PDF;
- **pandas**: organiza os dados em uma tabela;
- **re**: utiliza expressões regulares para localizar e separar as informações;
- **openpyxl**: permite salvar a tabela em formato Excel;
- **google.colab.files**: permite selecionar o PDF e baixar a planilha gerada.


In [ ]:
!pip -q install --no-deps pdfplumber
!pip -q install openpyxl

import re
import pdfplumber
import pandas as pd

from google.colab import files

print("Bibliotecas carregadas com sucesso!")


## Etapa 2 — Selecionar o arquivo PDF

Ao executar a célula abaixo, o Google Colab abrirá uma janela para selecionar o arquivo PDF no computador.

Depois do upload, o nome do arquivo será armazenado automaticamente na variável `arquivo_pdf`.


In [ ]:
uploaded = files.upload()

arquivo_pdf = next(iter(uploaded))

print("Arquivo carregado:", arquivo_pdf)


## Etapa 3 — Extrair o texto do PDF

O `pdfplumber` abre o documento página por página.

O texto encontrado em cada página é acrescentado à variável `texto_completo`.

Essa etapa ainda não organiza os dados: ela apenas transforma o conteúdo textual do PDF em uma sequência de texto que poderá ser analisada pelo Python.


In [ ]:
texto_completo = ""

with pdfplumber.open(arquivo_pdf) as pdf:
    for pagina in pdf.pages:
        texto = pagina.extract_text()

        if texto:
            texto_completo += texto + "\n"

print("Texto extraído com sucesso!")
print("Quantidade de caracteres:", len(texto_completo))


## Etapa 4 — Separar os cadastros individuais

Cada novo cadastro começa pelo campo **Nome:**.

A expressão regular abaixo procura esse marcador e divide o texto completo em registros individuais.

Assim, cada elemento da lista `registros` passa a representar uma pessoa.


In [ ]:
registros = re.split(
    r'(?i)(?=nome\s*:)',
    texto_completo
)

registros = [
    registro.strip()
    for registro in registros
    if registro.strip()
]

print("Quantidade de cadastros encontrados:", len(registros))


## Etapa 5 — Criar uma função para localizar cada característica

A função `extrair_campo()` recebe:

1. um padrão que indica qual informação deve ser procurada;
2. o texto do cadastro.

Quando encontra o campo, retorna seu conteúdo. Caso o campo não seja encontrado, retorna uma célula vazia.

O uso de `re.IGNORECASE` permite reconhecer diferenças entre letras maiúsculas e minúsculas.


In [ ]:
def extrair_campo(padrao, texto):

    resultado = re.search(
        padrao,
        texto,
        flags=re.IGNORECASE | re.MULTILINE
    )

    if resultado:
        return resultado.group(1).strip()

    return ""


## Etapa 6 — Extrair Nome, Apelido, Data, Endereço, CEP, Telefone e CPF

Agora cada cadastro é analisado individualmente.

O código também trata algumas situações especiais:

- separa o conteúdo entre `(aka ...)` em uma coluna chamada **Apelido**;
- reconhece tanto `Data de nascimento` quanto `Dt nasc`;
- reconhece tanto `Tel` quanto `Telefone`;
- procura o **CEP mesmo quando ele está na mesma linha do endereço**;
- depois de localizar o CEP, remove essa informação da coluna Endereço para evitar duplicação.


In [ ]:
dados = []

for registro in registros:

    # --------------------------------------------------------
    # NOME
    # --------------------------------------------------------
    nome_completo = extrair_campo(
        r'^nome\s*:\s*([^\n\r]+)',
        registro
    )

    nome = nome_completo
    apelido = ""

    # --------------------------------------------------------
    # APELIDO (AKA)
    # --------------------------------------------------------
    aka = re.search(
        r'\(aka\s+(.+?)\)',
        nome_completo,
        flags=re.IGNORECASE
    )

    if aka:
        apelido = aka.group(1).strip()

        nome = re.sub(
            r'\s*\(aka\s+.+?\)',
            '',
            nome_completo,
            flags=re.IGNORECASE
        ).strip()

    # --------------------------------------------------------
    # DATA DE NASCIMENTO
    # --------------------------------------------------------
    data_nascimento = extrair_campo(
        r'^(?:data\s+de\s+nascimento|dt\s*nasc)\s*:\s*([^\n\r]+)',
        registro
    )

    # --------------------------------------------------------
    # CEP
    # --------------------------------------------------------
    cep = extrair_campo(
        r'CEP\s*:\s*(\d{2}\.?\d{3}-?\d{3}|\d{5}-?\d{3})',
        registro
    )

    # --------------------------------------------------------
    # ENDEREÇO
    # --------------------------------------------------------
    endereco = extrair_campo(
        r'^endere[cç]o\s*:\s*([^\n\r]+)',
        registro
    )

    # Remove o CEP do endereço quando estiver na mesma linha
    endereco = re.sub(
        r'\s*CEP\s*:\s*\d{2}\.?\d{3}-?\d{3}',
        '',
        endereco,
        flags=re.IGNORECASE
    ).strip()

    # --------------------------------------------------------
    # TELEFONE
    # --------------------------------------------------------
    telefone = extrair_campo(
        r'^(?:tel|telefone)\s*:\s*([^\n\r]+)',
        registro
    )

    # --------------------------------------------------------
    # CPF
    # --------------------------------------------------------
    cpf = extrair_campo(
        r'^cpf\s*:\s*([\d.\-]+)',
        registro
    )

    # --------------------------------------------------------
    # ADICIONAR O CADASTRO À LISTA
    # --------------------------------------------------------
    dados.append({
        "Nome": nome,
        "Apelido": apelido,
        "Data de nascimento": data_nascimento,
        "Endereço": endereco,
        "CEP": cep,
        "Telefone": telefone,
        "CPF": cpf
    })

print("Extração dos campos concluída!")


## Etapa 7 — Transformar os dados em uma tabela

A lista de cadastros é convertida em um `DataFrame` do pandas.

CEP, telefone e CPF são mantidos como texto. Isso é importante porque esses campos podem começar com zero e não devem ser tratados como números para cálculos.


In [ ]:
df = pd.DataFrame(dados)

for coluna in ["CEP", "Telefone", "CPF"]:
    df[coluna] = df[coluna].fillna("").astype(str)

display(df)


## Etapa 8 — Verificar a quantidade de registros

Antes de salvar, podemos conferir quantas pessoas foram extraídas do PDF e quais colunas foram criadas.


In [ ]:
print("Total de pessoas:", len(df))
print("Colunas:", list(df.columns))


## Etapa 9 — Salvar a tabela em Excel

Nesta etapa o DataFrame é exportado para o arquivo:

`cadastro_extraido.xlsx`

Cada característica permanece em sua própria coluna, incluindo o **CEP separado do endereço**.


In [ ]:
arquivo_excel = "cadastro_extraido.xlsx"

df.to_excel(
    arquivo_excel,
    index=False,
    engine="openpyxl"
)

print("Arquivo Excel criado com sucesso:", arquivo_excel)


## Etapa 10 — Baixar o arquivo Excel

Ao executar a célula abaixo no Google Colab, o navegador iniciará o download da planilha gerada.


In [ ]:
files.download("cadastro_extraido.xlsx")
